In [ ]:
import json
import boto3
from kafka import KafkaConsumer
import os

In [ ]:
KAFKA_BROKER = "add your IP address:9092"
TOPIC = "demo_test"
BUCKET_NAME = "Amazon S3 Bucket Name"
SERIAL_FILE = "serial.txt"   # to save last serial

In [ ]:
s3 = boto3.client("s3")

In [ ]:
consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset="latest",
    group_id="stock-market-group"
)
print("Consumer started... Uploading to S3")

In [ ]:
if os.path.exists(SERIAL_FILE):
    with open(SERIAL_FILE, "r") as f:
        serial_no = int(f.read())
else:
    serial_no = 1

In [ ]:
for message in consumer:
    try:
        data = json.loads(message.value.decode("utf-8"))
    except json.JSONDecodeError:
        print("Skipping invalid JSON")
        continue

    file_name = f"stock_data/stock_data_{serial_no}.json"

    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=file_name,
        Body=json.dumps(data)
    )

    print("Uploaded:", file_name)

    serial_no += 1

    # Save serial so restart continues
    with open(SERIAL_FILE, "w") as f:
        f.write(str(serial_no))